# Data Wrangling — Dataset Stunting SIDIAS

Notebook ini berisi proses pembersihan dan transformasi data mentah dari file `dataset_stunting.csv`
menjadi dataset yang bersih dan siap dianalisis lebih lanjut.

Proses yang dilakukan mencakup:
1. Membaca dataset mentah
2. Menetapkan nama kolom yang sesuai
3. Memeriksa struktur dan tipe data
4. Konversi tipe data yang tidak sesuai
5. Menangani nilai yang hilang (missing value)
6. Standardisasi nilai kategori
7. Menyimpan dataset yang sudah bersih

## 1. Import Library

Mengimpor library yang dibutuhkan untuk proses wrangling:
- `pandas` untuk membaca dan memanipulasi data tabular
- `numpy` untuk operasi numerik dan penanganan nilai NaN
- `os` untuk mengatur path penyimpanan file

In [2]:
import pandas as pd
import numpy as np
import os

## 2. Membaca Dataset Mentah

Dataset mentah dibaca dari folder `data/raw`. File menggunakan pemisah titik koma (`;`)
dan baris pertama dilewati karena bukan header yang valid.
Hasilnya disimpan sementara sebelum diberi nama kolom yang sesuai.

In [3]:
# File menggunakan pemisah ';' dan tidak memiliki header
df_stunting = pd.read_csv('../data/raw/dataset_stunting.csv', sep=';', header=None, skiprows=1)

## 3. Menetapkan Nama Kolom

Dataset asli tidak memiliki header yang terbaca dengan benar, sehingga nama kolom
ditetapkan secara manual sesuai urutan data. Kolom yang tersedia mencakup identitas anak,
data antropometri (berat dan tinggi badan), serta status dan z-score untuk tiga indikator
gizi yaitu BB/U, TB/U, dan BB/TB.

In [4]:
# header yang sesuai dengan urutan kolom pada dataset
df_stunting.columns = [
    'id_anak', 'jenis_kelamin', 'usia_bulan', 'berat_badan', 'tinggi_badan',
    'status_bb_u', 'zscore_bb_u', 'status_tb_u', 'zscore_tb_u', 'status_bb_tb', 'zscore_bb_tb'
]

## 4. Memeriksa Struktur Data Awal

Pemeriksaan awal dilakukan untuk mengetahui jumlah baris, tipe data setiap kolom,
dan apakah ada nilai yang hilang. Dari hasil ini terlihat bahwa kolom `usia_bulan`
masih bertipe string dan kolom `status_bb_tb` memiliki 5 nilai yang hilang.

In [5]:
# Tampilkan informasi dasar tentang dataset
df_stunting.info()

<class 'pandas.DataFrame'>
RangeIndex: 40071 entries, 0 to 40070
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id_anak        40071 non-null  int64  
 1   jenis_kelamin  40071 non-null  str    
 2   usia_bulan     40071 non-null  str    
 3   berat_badan    40071 non-null  float64
 4   tinggi_badan   40071 non-null  float64
 5   status_bb_u    40071 non-null  str    
 6   zscore_bb_u    40071 non-null  float64
 7   status_tb_u    40071 non-null  str    
 8   zscore_tb_u    40071 non-null  float64
 9   status_bb_tb   40066 non-null  str    
 10  zscore_bb_tb   40071 non-null  float64
dtypes: float64(5), int64(1), str(5)
memory usage: 3.4 MB


## 5. Konversi Tipe Data

Kolom `usia_bulan` masih bertipe string sehingga perlu dikonversi ke numerik.
Nilai yang tidak bisa dikonversi akan otomatis berubah menjadi NaN dan ditangani
di langkah berikutnya. Parameter `errors='coerce'` digunakan supaya proses
konversi tidak berhenti meski ada nilai yang tidak valid.

In [6]:
# konversi kolom usia_bulan ke tipe numerik
df_stunting['usia_bulan'] = pd.to_numeric(df_stunting['usia_bulan'], errors='coerce')

## 6. Memeriksa Struktur Data Setelah Konversi

Setelah konversi, kolom `usia_bulan` sudah bertipe float64. Namun muncul 2 nilai NaN
baru yang sebelumnya berupa string tidak valid. Nilai ini akan ditangani di langkah berikutnya.

In [7]:
# Tampilkan informasi dasar tentang dataset setelah konversi
df_stunting.info()

<class 'pandas.DataFrame'>
RangeIndex: 40071 entries, 0 to 40070
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id_anak        40071 non-null  int64  
 1   jenis_kelamin  40071 non-null  str    
 2   usia_bulan     40069 non-null  float64
 3   berat_badan    40071 non-null  float64
 4   tinggi_badan   40071 non-null  float64
 5   status_bb_u    40071 non-null  str    
 6   zscore_bb_u    40071 non-null  float64
 7   status_tb_u    40071 non-null  str    
 8   zscore_tb_u    40071 non-null  float64
 9   status_bb_tb   40066 non-null  str    
 10  zscore_bb_tb   40071 non-null  float64
dtypes: float64(6), int64(1), str(4)
memory usage: 3.4 MB


## 7. Menangani Nilai yang Hilang (Missing Value)

Ditemukan dua jenis nilai hilang yang ditangani dengan cara berbeda:

- `usia_bulan` (2 baris hilang, tipe numerik) → diisi dengan nilai median kolom tersebut
  supaya tidak menggeser distribusi data secara signifikan
- `status_bb_tb` (5 baris hilang, tipe kategorik) → baris dihapus karena tidak bisa
  diimputasi tanpa menimbulkan bias pada label kategori

Sebelumnya nilai string `#N/A` yang ada di dataset juga diganti menjadi NaN
agar pandas bisa mengenalinya sebagai nilai hilang yang sesungguhnya.

In [8]:
# replace nilai '#N/A' dengan NaN untuk penanganan data yang hilang
df_stunting.replace('#N/A', np.nan, inplace=True)

# usia_bulan → isi median (numerik, 2 baris hilang)
df_stunting['usia_bulan'] = df_stunting['usia_bulan'].fillna(df_stunting['usia_bulan'].median())

# status_bb_tb → drop baris (kategorik, 5 baris hilang)
df_stunting.dropna(subset=['status_bb_tb'], inplace=True)
df_stunting.reset_index(drop=True, inplace=True)

## 8. Standardisasi Nilai Kolom Jenis Kelamin

Kolom `jenis_kelamin` menggunakan kode singkat `M` dan `F` yang kurang deskriptif.
Nilai ini dipetakan ke `Laki-laki` dan `Perempuan` supaya lebih mudah dibaca
di tahap analisis dan visualisasi selanjutnya.

In [9]:
# Mapping jenis_kelamin dari 'M' dan 'F' ke 'Laki-laki' dan 'Perempuan'
mapping_gender = {'M': 'Laki-laki', 'F': 'Perempuan'}
df_stunting['jenis_kelamin'] = df_stunting['jenis_kelamin'].map(mapping_gender)

## 9. Standardisasi Nilai Kolom Status Gizi

Kolom status gizi masih menggunakan istilah dalam bahasa Inggris seperti `Stunted`,
`Thin`, dan `Obese`. Semua nilai ini dipetakan ke padanan bahasa Indonesia supaya
konsisten dengan konteks proyek dan lebih mudah dipahami oleh pengguna sistem.

In [10]:
# Mapping status gizi dan stunting ke bahasa Indonesia
mapping_status = {
    'Normal': 'Normal',
    'Stunted': 'Stunting',
    'Not Stunted': 'Tidak Stunting',
    'Malnutrition': 'Gizi Buruk',
    'Underfed': 'Gizi Kurang',
    'Thin': 'Kurus',
    'Very Thin': 'Sangat Kurus',
    'Obese': 'Obesitas',
    'Overnutrition': 'Gizi Lebih'
}

In [11]:
# Terapkan mapping status pada kolom status_bb_u, status_tb_u, dan status_bb_tb
kolom_status = ['status_bb_u', 'status_tb_u', 'status_bb_tb']
for col in kolom_status:
    df_stunting[col] = df_stunting[col].replace(mapping_status)

## 10. Memastikan Tipe Data Final

Memastikan kolom `usia_bulan` benar-benar bertipe float setelah semua proses
imputasi selesai, supaya tidak ada masalah tipe data saat dataset digunakan
di tahap EDA maupun pemodelan.

In [12]:
# Pastikan kolom usia_bulan sudah dalam tipe numerik
df_stunting['usia_bulan'] = df_stunting['usia_bulan'].astype(float)

## 11. Verifikasi Hasil Pembersihan

Menampilkan beberapa baris pertama untuk memastikan semua transformasi sudah berjalan
dengan benar. Kolom `jenis_kelamin` sudah terbaca dalam bahasa Indonesia dan
kolom `status_tb_u` sudah menggunakan istilah yang sesuai.

In [13]:
# Tampilkan beberapa baris pertama untuk memastikan data sudah bersih dan terstruktur dengan baik
print(df_stunting[['id_anak', 'jenis_kelamin', 'usia_bulan', 'status_tb_u']].head())

   id_anak jenis_kelamin  usia_bulan status_tb_u
0        1     Perempuan        54.0    Stunting
1        2     Laki-laki        44.0    Stunting
2        3     Laki-laki        57.0    Stunting
3        4     Laki-laki        26.0    Stunting
4        5     Perempuan        59.0    Stunting


## 12. Ringkasan Dataset Setelah Wrangling

Pemeriksaan akhir untuk memastikan tidak ada lagi nilai yang hilang di seluruh kolom
dan semua tipe data sudah sesuai. Dataset akhir terdiri dari 40.066 baris bersih
yang siap dilanjutkan ke tahap EDA dan feature engineering.

In [14]:
df_stunting.info()

<class 'pandas.DataFrame'>
RangeIndex: 40066 entries, 0 to 40065
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id_anak        40066 non-null  int64  
 1   jenis_kelamin  40066 non-null  str    
 2   usia_bulan     40066 non-null  float64
 3   berat_badan    40066 non-null  float64
 4   tinggi_badan   40066 non-null  float64
 5   status_bb_u    40066 non-null  str    
 6   zscore_bb_u    40066 non-null  float64
 7   status_tb_u    40066 non-null  str    
 8   zscore_tb_u    40066 non-null  float64
 9   status_bb_tb   40066 non-null  str    
 10  zscore_bb_tb   40066 non-null  float64
dtypes: float64(6), int64(1), str(4)
memory usage: 3.4 MB


## 13. Menyimpan Dataset yang Sudah Bersih

Dataset yang sudah melalui seluruh proses pembersihan disimpan ke folder `data/processed`
dengan nama `dataset_stunting_clean.csv`. File ini yang akan digunakan sebagai input
di notebook EDA dan Feature Engineering selanjutnya.

In [15]:
# Simpan dataset yang sudah dibersihkan ke file baru
save_path = os.path.join('../data/processed', 'dataset_stunting_clean.csv')
df_stunting.to_csv(save_path, index=False)

## Kesimpulan Proses Wrangling

Proses pembersihan data berhasil diselesaikan dengan ringkasan sebagai berikut:

- Data awal berjumlah 40.071 baris dengan 11 kolom
- 2 nilai hilang di kolom `usia_bulan` diisi dengan nilai median
- 5 baris dengan nilai hilang di kolom `status_bb_tb` dihapus
- Kolom `usia_bulan` berhasil dikonversi dari string ke numerik
- Kolom `jenis_kelamin` distandarisasi dari kode M/F ke label lengkap
- Seluruh label status gizi diterjemahkan ke bahasa Indonesia
- Dataset akhir bersih berjumlah 40.066 baris tanpa missing value

Dataset siap digunakan untuk tahap Exploratory Data Analysis (EDA).